# មេរៀន 09 - លំនាំការរចនាអំពីការយល់ដឹងពីការគិត


## ការតំឡើង

កំណត់ត្រានេះបង្ហាញពីលំនាំការរចនា Metacognition ដោយប្រើ Microsoft Agent Framework។

**តម្រូវការមុនៈ**
- ការដាក់ចេញ Azure OpenAI ដែលបានកំណត់តាមអថេរបរិស្ថាន
- Azure CLI បានផ្ទៀងផ្ទាត់ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Metacognition គឺជាអ្វី?

Metacognition គឺជា **ការគិតអំពីការគិត**។ ក្នុងបរិបទនៃភ្នាក់ងារ AI, វាមានន័យថាបង្កើតភ្នាក់ងារដែលអាច:

- **ធ្វើការត្រួតពិនិត្យខ្លួនឯង** លើលទ្ធផល និងដំណើរការវិភាគរបស់ពួកគេ
- **រកឃើញកំហុស** និងស្ដារឡើងវិញដោយយ៉ាងសមរម្យ ជំនួសការបរាជ័យដោយស្ងាត់
- **វាយតម្លៃ** ថាតើចម្លើយរបស់ពួកគេពេញលេញ និងមានប្រយោជន៍ឬទេ
- **បត់បែន** យុទ្ធសាស្ត្ររបស់ពួកគេនៅពេលវិធីដំបូងមិនដំណើរការ (ឧ. ត្រឡប់ទៅប្រព័ន្ធបម្រុងទុក)

ភ្នាក់ងារមេតាគិតមិនត្រឹមតែឆ្លើយសំណួរប៉ុណ្ណោះទេ — វាត្រួតពិនិត្យសមត្ថភាពខ្លួនឯង និងកែសម្រួលភ្លាមៗ។


## ឧបករណ៍ចម្បង និង ឧបករណ៍បម្រុង

លំនាំមួយដែលពេញនិយមនៅក្នុងមេតាឧត្តមវិទ្យាគឺ **យុទ្ធសាស្ត្រជំនួស**។ ភ្នាក់ងារ​ព្យាយាម​ប្រើឧបករណ៍ចម្បងជាលើកដំបូង; ប្រសិនបើវាបរាជ័យ (ឧ. កំហុស 404), ភ្នាក់ងារនោះនឹងទទួលស្គាល់ការបរាជ័យ និងផ្លាស់ប្តូរទៅឧបករណ៍បម្រុងយ៉ាងច្បាស់លាស់។

នេះស្រដៀងនឹងប្រព័ន្ធក្នុងពិភពពិតដែលសេវាកម្មចម្បងអាចមិនមានដំណើរការ ហើយភ្នាក់ងារត្រូវតែវាយតម្លៃបញ្ហាផ្ទាល់ខ្លួនមុនពេលជ្រើសផ្លូវជំនួស។

ខាងក្រោមនេះយើងកំណត់ឧបករណ៍ស្វែងរកចំណតហោះហើរចំនួនពីរ៖
- **ចម្បង** — គ្របដណ្ដប់ Paris, Tokyo, និង Barcelona
- **បម្រុង** — គ្របដណ្ដប់ Berlin, Sydney, និង New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ភ្នាក់ងារត្រលប់មើលខ្លួនឯង ជាមួយការស្ដារកំហុស

ភ្នាក់ងារខាងក្រោមត្រូវបានណែនាំឱ្យព្យាយាមប្រើប្រព័ន្ធហោះចម្បងជាលើកដំបូង ស្គាល់កំហុស ហើយយ៉ាងច្បាស់ថយចុះទៅប្រព័ន្ធបម្រុង។ បន្ទាប់ពីការឆ្លើយតបរាល់មួយ វានឹងធ្វើការវាយតម្លៃខ្លីថាតើវាបានឆ្លើយសំណួររបស់អ្នកប្រើប្រាស់ពេញលេញឬអត់។


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## គំរូការវាយតម្លៃខ្លួនឯង

មុខម្ខាងទៀតនៃការយល់ដឹងអំពីការយល់ដឹង (metacognition) គឺ **ការវាយតម្លៃខ្លួនឯង**: តួអង្គផ្សេងម្នាក់ (ឬតួអង្គដដែលនៅក្នុងដំណើរការជាដំណាក់កាលទីពីរ) ពិនិត្យចម្លើយមួយសម្រាប់ភាពពេញលេញ ភាពត្រឹមត្រូវ និងភាពមានប្រយោជន៍។

ខាងក្រោម យើងបង្កើតភ្នាក់ងារ `ResponseEvaluator` ដែលផ្តល់ពិន្ទុចម្លើយរបស់ភ្នាក់ងារធ្វើដំណើរលើបីវិមាត្រ។


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## សង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀបបង្កើត **ភ្នាក់ងារយល់ដឹងអំពីខ្លួនឯង** ដោយប្រើ Microsoft Agent Framework:

- **ការពិចារណាផ្ទាល់​ខ្លួន**: ភ្នាក់ងារ​ដែលត្រួតពិនិត្យដំណើរការគិតរបស់ខ្លួន និងធ្វើការប្រាស្រ័យយ៉ាងច្បាស់អំពីអ្វីដែលបានកើតขึ้น។
- **ការស្ដារកំហុសជាមួយវិធីជំនួស**: លំនាំឧបករណ៍បឋម + ជំនួស ដែលភ្នាក់ងារសម្គាល់កំហុស (ឧ. កំហុស 404) ហើយព្យាយាមប្រភពជំនួសដោយស្វ័យប្រវត្តិ។
- **ការវាយតម្លៃដោយខ្លួនឯង**: ភ្នាក់ងារវាយតម្លៃឯករាជ្យមួយដែលផ្តល់ពិន្ទុចំពោះចម្លើយសម្រាប់ភាពពេញលេញ ភាពត្រឹមត្រូវ និងភាពមានប្រយោជន៍។

លំនាំទាំងនេះបង្កើតឲ្យភ្នាក់ងារមានភាពរឹងមាំ ភាពច្បាស់លាស់ និងគួរឱ្យទុកចិត្ត — គុណលក្ខណ៍សំខាន់សម្រាប់ការដាក់ចេញក្នុងបរិបទផលិតកម្ម។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការមិនទទួលខុសត្រូវ**:
ឯកសារ​នេះ​ត្រូវ​បាន​បកប្រែ​ដោយ​ប្រើ​សេវាកម្ម​បកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលដែលយើងខិតខំនូវភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថាការបកប្រែដោយស្វ័យប្រវត្តិកុំព្យូទ័រអាចមានកំហុស ឬមិនត្រឹមត្រូវ។ ឯកសារ​ដើម​នៅ​ក្នុង​ភាសាដើម​គួរត្រូវបានគេចាត់ទុកថាជាប្រភពយោងដែលមានអំណាច។ សម្រាប់ព័ត៌មានសំខាន់ៗ ណែនាំឱ្យធ្វើការបកប្រែដោយអ្នកបកប្រែដែលជាជំនាញ។ យើងមិនទទួលខុសត្រូវ​ចំពោះ​ការ​យល់ច្រឡំ ឬ​ការ​បកប្រែខុសណាមួយ​ដែល​កើតឡើង​ពី​ការ​ប្រើប្រាស់​ការ​បកប្រែ​នេះ​ទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
